<div style="display:flex">
    <img src="https://bse.eu/sites/default/files/bse_logo_small.png" alt="Logo 1">
</div>

# Data Preparation - Practice

In [ ]:
# Installation of the required packages
# conda create -n dl-applications
!pip install --upgrade pip setuptools wheel
!pip install jupyter ipykernel
!pip install --upgrade numpy 
!pip install imutils tqdm matplotlib pandas numpy opencv-python scipy

#!python -m ipykernel install --user --name $ENVNAME  ### CHANGE 'ENVNAME' with your virtual environment name e.g. dl-application-flood

In [ ]:
# Importing the required packages

# Standard library
import os
import re
import random
import shutil
import urllib.request
from pathlib import Path
from typing import List, Tuple

# Third-party
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from imutils import paths
from tqdm import tqdm

In [ ]:

DATA_URL = "https://www.dropbox.com/scl/fi/2uky8nbzkinmjcsliugna/train.zip?rlkey=190u30cr3hjy4zi7dmolvnsn4&st=s03k2qoe&dl=1"

def download_data(
    url: str = DATA_URL,
    extract_path: Path = Path("train"),
    force: bool = False,
) -> Path:
    """Download and extract the training data.

    Args:
        url: Source URL for the zip archive.
        extract_path: Destination directory for the extracted files.
        force: If True, re-download even if extract_path already exists.

    Returns:
        Path to the extracted directory.

    Raises:
        urllib.error.URLError: If the download fails.
        shutil.ReadError: If the archive cannot be unpacked.
    """
    extract_path = Path(extract_path)

    if extract_path.exists() and not force:
        print(f"Data already present at '{extract_path}', skipping download.")
        return extract_path

    zip_path = extract_path.with_suffix(".zip")

    try:
        print("Downloading...")
        urllib.request.urlretrieve(url, zip_path)

        print("Extracting...")
        shutil.unpack_archive(zip_path, extract_path)
    finally:
        if zip_path.exists():
            zip_path.unlink()

    print(f"Extracted contents of '{extract_path}':")
    for entry in sorted(extract_path.iterdir()):
        print(f" - {entry.name}")

    return extract_path

# Step 1: Download the data

The download will take few minutes

In [ ]:
data_dir = download_data()
os.chdir('train')

# Step 2: Investigate the data

Please read carefully the documentation of the [ETCI 2021 Competition on Flood Detection](https://nasa-impact.github.io/etci2021/) while downloading the train data. Then, inspect the data folders from the File Explorer to get a sense of the dataset structure. 

Relevant questions you may want to investigate online (this exercise aims to improve your skills when approaching a  project in a new domain):
- What is a GeoTIFF file? 
- What is an ESA Copernicus Sentinel-1 image? 
- What do VV and VH stand for?
- ... any other question that will help you better understand the problem. 

**Before starting:** In 1-2 sentences, explain why SAR (Synthetic Aperture Radar)
  imagery is particularly useful for flood detection compared to optical imagery
  (e.g. Landsat). Hint: think about cloud cover and water surface reflection.

In [ ]:
all_image_paths = list(paths.list_images("train")) 
print(f"Total images: {int(len(all_image_paths))/4}")

From [here](https://nasa-impact.github.io/etci2021/): "The contest dataset is composed of 66,810 (33,405 x 2 VV & VH polarization) tiles of 256×256 pixels, distributed respectively across the training, validation and test sets as follows: 33,405, 10,400, and 12,348 tiles for each polarization."

For a given image id (e.g. `nebraska_20170309t002110`), its corresponding ground-truths i.e. the segmentation maps are present in either of these two folders: `water_body_label` and `flood_label`. 

In [ ]:
def get_image_ids(all_image_paths):
    """ Extract the image IDs from the image paths
    Args:
        all_image_paths: List of image paths
    Returns:
        image_ids: Set of image IDs
    Raises:
        Exception: An error occurred while extracting image IDs
    """
    # Extract the image IDs from the image paths
    try:
        image_ids = {path.split(os.sep)[1] for path in all_image_paths}
        return image_ids
    # Handle exceptions
    except Exception as e:
        print(f'An error occurred while extracting image IDs: {e}')
        return set()

In [ ]:
image_ids=get_image_ids(all_image_paths)
image_ids

In [ ]:
print(len(image_ids))

Now, let's investigate how these IDs are distributed. 

1. Write a function `get_image_paths` that takes the `image_id` as input and returns the path of all four sub-folders `flood_image_path`, `water_body_path`, `vh_image_paths`, and `vv_image_paths`. 

In [ ]:
from pathlib import Path
from typing import Tuple, List

def get_image_paths(image_id: str, root: Path = Path("train")) -> Tuple[List[Path], List[Path], List[Path], List[Path]]:
    """Get the paths to the PNG images for a specific image ID.

    Args:
        image_id: The image ID (folder name under <root>).
        root: Path to the dataset root. Defaults to 'train'.

    Returns:
        A tuple of four lists:
            - flood_image_paths: paths to the flood label images
            - water_body_paths:  paths to the water body label images
            - vh_image_paths:    paths to the VH images
            - vv_image_paths:    paths to the VV images
    """
    base = Path(root) / image_id / "tiles"
    subdirs = ("flood_label", "water_body_label", "vh", "vv")

    return tuple(sorted((base / name).glob("*")) for name in subdirs)

2. Do all the IDs have the same amount of images present?

In [ ]:
distribution_dict = {}

for id in tqdm(image_ids):
    distribution_dict[id] = {}
    flood_image_paths, water_body_paths, vh_image_paths, vv_image_paths = \
        get_image_paths(id)

    distribution_dict[id]["flood_images"] = len(flood_image_paths)
    distribution_dict[id]["water_body_images"] = len(water_body_paths)
    distribution_dict[id]["vh_images"] = len(vh_image_paths)
    distribution_dict[id]["vv_images"] = len(vv_image_paths)

distribution_df = pd.DataFrame.from_dict(distribution_dict).T
assert len(distribution_df) == len(image_ids)
distribution_df

No huge distribution skews noticed. But for `bangladesh_20170314t115609` there is a mismatch between the number of flood image maps and the number of VV images.

## Visualization

Now, let's write a utility function that would return the images belonging to the format - `<region>_<datetime>*_x-*_y-*.png`. 


<p align="center">
<img src=https://i.ibb.co/mCZp6X4/image.png></ing>
</p>

However, 

> We expect participants to provide a binary segmentation of the region of interest (ROI), (i.e. 256x256 pixels) as a numpy array with the byte (uint8) data type:
**1: Flood region, 0: Not flood region**.

In [ ]:
# https://stackoverflow.com/a/2669120/7636462
def sorted_nicely(l: List[str]) -> List[str]:
    """
    Sort the given iterable alphanumerically.

    Args:
    l: List of strings to be sorted.

    Returns:
    Sorted list of strings.
    """
    def convert(text):
        return int(text) if text.isdigit() else text

    def alphanum_key(key):
        return [convert(c) for c in re.split('([0-9]+)', key)]

    return sorted(l, key=alphanum_key)

In [ ]:
all_image_paths = sorted_nicely(all_image_paths)
all_image_paths

What is `.ipynb_checkpoints` doing here? **It might not be present for all depending on the way your computer is storing Jupyter Notbeook checkpoints. However, it explains why  `bangladesh_20170314t115609` had a mismatch in the number of images and labels.**

3. Write a function `filter_image_paths` to split `all_image_paths` in the `vv_image_paths`,`vh_image_paths`,`flood_image_paths`,`water_body_label_paths`

In [ ]:
from typing import Iterable, List

def filter_image_paths(all_image_paths: Iterable[Path | str], keyword: str) -> List[Path]:
    """Filter image paths whose path contains the given folder keyword.

    Excludes Jupyter checkpoint artifacts.

    Args:
        all_image_paths: Iterable of image paths.
        keyword: Folder name to match (e.g., 'vv', 'vh', 'flood_label', 'water_body_label').

    Returns:
        List of matching paths.
    """
    return [
        Path(p) for p in all_image_paths
        if keyword in Path(p).parts and ".ipynb_checkpoints" not in Path(p).parts
    ]


vv_paths     = filter_image_paths(all_image_paths, "vv")
vh_paths     = filter_image_paths(all_image_paths, "vh")
flood_paths  = filter_image_paths(all_image_paths, "flood_label")
water_paths  = filter_image_paths(all_image_paths, "water_body_label")

print(f"Flood label paths:      {len(flood_paths)}")
print(f"Water body label paths: {len(water_paths)}")
print(f"VV image paths:         {len(vv_paths)}")
print(f"VH image paths:         {len(vh_paths)}")

Note that accessing the files from different folders just using the index is working only and only if the files are stored in equal numbers in each folder and are ordered in the same way. Let's print what is stored in each folder for the index = 10. Asserting the naming convention is key!

In [ ]:
print("VV image path index 10",vv_paths[10])
print("VH body path index 10",vh_paths[10])
print("Flood image path index 10",flood_paths[10])
print("Water image path index 10",water_paths[10])

In [ ]:
from pathlib import Path
from typing import Sequence, Optional
import matplotlib.pyplot as plt

def show_all_four_images(
    filenames: Sequence[Path | str],
    titles: Sequence[str],
    columns: int = 4,
    figsize_per_image: tuple[int, int] = (5, 5),
    suptitle: Optional[str] = None,
    cmap: Optional[str] = "gray",
) -> None:
    """Display a grid of images with titles.

    Args:
        filenames: Image paths to display.
        titles: Title for each image (same length as filenames).
        columns: Number of columns in the grid. Defaults to 4.
        figsize_per_image: (width, height) in inches for each subplot. Defaults to (5, 5).
        suptitle: Optional figure title. If None, uses the first image's parent ID.
        cmap: Colormap for single-channel images. Ignored for RGB. Defaults to 'gray'.
    """
    if len(filenames) != len(titles):
        raise ValueError(
            f"Mismatched lengths: {len(filenames)} filenames vs {len(titles)} titles."
        )
    if not filenames:
        raise ValueError("No filenames provided.")

    paths = [Path(f) for f in filenames]
    n = len(paths)
    rows = (n + columns - 1) // columns

    fig, axes = plt.subplots(
        rows, columns,
        figsize=(figsize_per_image[0] * columns, figsize_per_image[1] * rows),
        squeeze=False,
    )

    # Default suptitle: the image_id (folder two levels up from a tile file)
    if suptitle is None:
        try:
            suptitle = paths[0].parents[2].name
        except IndexError:
            suptitle = ""
    if suptitle:
        fig.suptitle(suptitle, size=16)

    for ax, path, title in zip(axes.flat, paths, titles):
        img = plt.imread(path)
        ax.imshow(img, cmap=cmap if img.ndim == 2 else None)
        ax.set_title(title)
        ax.axis("off")

    # Hide any unused axes in the final row
    for ax in axes.flat[n:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
titles = ["VV","VH" , "Land or water before flood/Water body image" ,"After Flood/flood image"]
#Documentation available: https://numpy.org/doc/stable/reference/random/generated/numpy.random.random_sample.html#numpy.random.random_sample
random_index =  random.sample(range(0, len(vv_paths)), 20) 
for i in random_index:
    show_all_four_images([vv_paths[i], vh_paths[i], water_paths[i], flood_paths[i] ] , titles)

**Some noise found (from an earlier iteration)**:

* https://ibb.co/m6x9f1S
* https://ibb.co/rfWtJy7

How in an all-white image, any segmentation map is present? 

### Displaying the RGB composite

From [here](https://sentinel.esa.int/web/sentinel/user-guides/sentinel-1-sar/product-overview/polarimetry):

> The composite RGB (colour) image on the right was created using the VV channel for red, VH channel for green and the ratio $|VV| / |VH|$ for blue.

In [ ]:
from pathlib import Path
from PIL import Image

def show_all_combined_images(i: int, titles: List[str], vv_image_paths: List[str], vh_image_paths: List[str], water_body_label_paths: List[str], flood_image_paths: List[str]) -> None:
    """
    Display three images in a row.

    Args:
        i: Index of the image.
        titles: List of titles for the images.
        vv_image_paths: List of VV image paths.
        vh_image_paths: List of VH image paths.
        water_body_label_paths: List of water body label image paths.
        flood_image_paths: List of flood image paths.

    Returns:
        None
    """
    columns = 3
    eps = 1e-07

    # Load and process the VV and VH images
    red, _, _ = Image.open(vv_image_paths[i]).split()
    red = np.asarray(red)
    _, green, _ = Image.open(vh_image_paths[i]).split()
    green = np.asarray(green)

    # Create the blue channel
    blue = np.abs(red) / (np.abs(green) + eps)
    blue = (blue * 255).astype(np.uint8)
    rgb = Image.fromarray(np.dstack((red, green, blue)))

    # Load the water body and flood images
    images = [rgb]
    images.append(plt.imread(water_body_label_paths[i]))
    images.append(plt.imread(flood_image_paths[i]))

    # Display the images
    plt.figure(figsize=(20, 10))
    plt.suptitle(Path(vv_image_paths[i]).parents[2].name, size=16)
    for j, image in enumerate(images):
        ax = plt.subplot(1, columns, j + 1)
        ax.set_title(titles[j])
        ax.imshow(image)
        ax.axis('off')  # Hide axes for better visualization

    plt.tight_layout()
    plt.show()

In [ ]:
titles = ["Combined" , "Land or water before flood/Water body image" ,"After Flood/flood image"]
for i in random_index:
    show_all_combined_images(i, titles, vv_paths, vh_paths, water_paths, flood_paths)

## Observation


Let's identify the type of noise present in the dataset and manually inspect a few of these tiles. Determine potentially noisy label indexes from the plot above, or use the pre-selected noisy label indexes [19311, 23116, 24354, 10041].

In [ ]:
noisy_labels_indexes=[19311, 23116, 24354, 10041]
print("Potentially noise labels:", noisy_labels_indexes)
titles = ["Combined" , "Land or water before flood/Water body image" ,"After Flood/flood image"]
for i in noisy_labels_indexes:
    show_all_combined_images(i, titles, vv_paths, vh_paths, water_paths, flood_paths)

4. Answer the following questions:

**Q1:** What shall I expect from the "Water" and the "Flood" tiles? Should the Flood Extent be more extensive than the Water Extent? If so, why?

<details>
<summary>Show answer</summary>

The Water Extent seems derived from some external dataset provided by the permanent waters (rivers, lakes) from the area. The Flood Extent instead seems to be related to a specific flood event.

</details>

**Q2:** Why are some images cut?

<details>
<summary>Show answer</summary>

This is likely because the satellite imagery isn't aligned with the x and y axes, and the orthogonal clipping caused the visible cropping.

</details>

**Q3:** What to do with the tiles with noisy labels?

<details>
<summary>Show answer</summary>

Processing them to produce a clean dataset.

</details>

5. Load the data with both [Image.open()](https://pillow.readthedocs.io/en/stable/reference/Image.html) and [mpimg.imread()](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imread.html). Indentify the differences between the two methods get understand the data distribution answering some of these questions: 

In [ ]:
#Let's investigate the first noisy labels:
index=noisy_labels_indexes[0]
print("Noisy Label Index:",index)

vh_image=Image.open(vh_paths[index])
vv_image=Image.open(vv_paths[index])
flood_image=Image.open(flood_paths[index])
water_label=Image.open(water_paths[index])

print("Unique Values in VV",np.unique(vv_image))
print("Unique Values in VH",np.unique(vh_image))
print("Unique Values in water body label",np.unique(water_label))
print("Unique Values in flood image",np.unique(flood_image))

In [ ]:

import matplotlib.image as mpimg
vh_image=mpimg.imread(vh_paths[index])
vv_image=mpimg.imread(vv_paths[index])
flood_image=mpimg.imread(flood_paths[index])
water_body_label=mpimg.imread(water_paths[index])


print("Unique Values in VV",np.unique(vv_image))
print("Unique Values in VH",np.unique(vh_image))
print("Unique Values in water body label",np.unique(water_body_label))
print("Unique Values in flood image",np.unique(flood_image))

**Q1:** What are the values of the blank images?

<details>
<summary>Show answer</summary>

The blank images have value 255. The water/flood pixels are set to 255 and the background in the labels are set to 0.

</details>

**Q2:** Do all images have an informative label?

<details>
<summary>Show answer</summary>

There are images without labels. If an image contains no flood or water, it is considered a background image. However, if an image does contain flood or water, the absence of a label is incorrect, which can lead to errors during training.

</details>

## Cleaning Procedure:

6. Write a function `find_empty_images` that  takes as input three lists of file paths to VV images, VH images, and flood label images respectively. It identifies and returns two lists: one containing the paths of blank VV and VH images (all white), and another containing the paths of flood label images that have no flood/water extent (all background).

Useful questions: 
* Can we discard the blank images (all white ones under `Combined` and their respective labels)? 
* Can we discard the ones that have a flood/water extent corresponding to a region that looks outside the image? 


In [ ]:
from pathlib import Path
from typing import Sequence, Tuple, List
import numpy as np
from PIL import Image
from tqdm import tqdm


def find_empty_images(
    vv_paths: Sequence[Path | str],
    vh_paths: Sequence[Path | str],
    flood_paths: Sequence[Path | str],
    blank_value: int = 255,
    verbose: bool = True,
) -> Tuple[List[Path], List[Path]]:
    """Find blank VV/VH images and empty flood label masks.

    A VV image is considered blank if both the VV and the matching VH tile are
    uniformly equal to `blank_value`. A flood label is considered empty if all
    its pixels are zero.

    Args:
        vv_paths: VV image paths.
        vh_paths: VH image paths
        flood_paths: Flood label paths.
        blank_value: Pixel value indicating a blank tile. Defaults to 255.
        verbose: If True, print summary statistics.

    Returns:
        (empty_vv_paths, empty_flood_label_paths)
    """
    n = len(vv_paths)
    if not (n == len(vh_paths) == len(flood_paths)):
        raise ValueError(
            f"Path lists must be the same length: "
            f"vv={n}, vh={len(vh_paths)}, flood={len(flood_paths)}"
        )

    empty_vv_paths: List[Path] = []
    empty_flood_label_paths: List[Path] = []

    for vv_p, vh_p, flood_p in tqdm(
        zip(vv_paths, vh_paths, flood_paths),
        total=n,
        desc="Checking tiles",
    ):
        vv = np.asarray(Image.open(vv_p))
        vh = np.asarray(Image.open(vh_p))
        flood = np.asarray(Image.open(flood_p))
        # Logic here: all the VV and VH pixels are =black_value
        if np.all(vv == blank_value) and np.all(vh == blank_value):
            empty_vv_paths.append(Path(vv_p))
        #.any() is a NumPy array method that returns True if any element is truthy (non-zero, non-False, non-empty),
        # and False if every element is zero/falsy. 
        if not flood.any():
            empty_flood_label_paths.append(Path(flood_p))

    if verbose:
        n_blank = len(empty_vv_paths)
        n_empty_flood = len(empty_flood_label_paths)
        print(f"Blank VV tiles:        {n_blank} / {n}  ({100 * n_blank / n:.2f}%)")
        print(f"Empty flood labels:    {n_empty_flood} / {n}  ({100 * n_empty_flood / n:.2f}%)")

    return empty_vv_paths, empty_flood_label_paths

In [ ]:
#Take few minutes
empty_vv_paths, empty_flood_label_paths = find_empty_images(vv_paths, vh_paths, flood_paths)  


## Step 3: Class Distribution

In [ ]:
from pathlib import Path
from typing import Iterable, Tuple
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm


def analyze_flood_images(
    flood_paths: Iterable[Path | str],
    expected_shape: Tuple[int, int] = (256, 256),
    verbose: bool = True,
) -> Tuple[int, int, int, int]:
    """Count water and background pixels and tiles across flood label images.

    Args:
        flood_paths: Iterable of flood label image paths.
        expected_shape: Expected (height, width) of each tile.
        verbose: If True, print summary statistics.

    Returns:
        (total_water_pixels, total_background_pixels, total_water_tiles, total_background_tiles)
    """
    total_water_pixels = 0
    total_background_pixels = 0
    total_water_tiles = 0
    total_background_tiles = 0
    pixels_per_tile = expected_shape[0] * expected_shape[1]

    for filename in tqdm(flood_paths, desc="Analyzing flood images"):
        img = plt.imread(filename)
        mask = img[..., 0] if img.ndim == 3 else img

        if mask.shape != expected_shape:
            raise ValueError(f"Unexpected shape {mask.shape} for {filename}")

        water_pixels = int((mask > 0).sum())
        total_water_pixels += water_pixels
        total_background_pixels += pixels_per_tile - water_pixels

        if water_pixels > 0:
            total_water_tiles += 1
        else:
            total_background_tiles += 1

    if verbose:
        total_pixels = total_water_pixels + total_background_pixels
        total_tiles = total_water_tiles + total_background_tiles
        print(f"Water tiles:       {total_water_tiles:>6} ({100 * total_water_tiles / total_tiles:.2f}%)")
        print(f"Background tiles:  {total_background_tiles:>6} ({100 * total_background_tiles / total_tiles:.2f}%)")
        print(f"Water pixels:      {total_water_pixels:>10} ({100 * total_water_pixels / total_pixels:.2f}%)")
        print(f"Background pixels: {total_background_pixels:>10} ({100 * total_background_pixels / total_pixels:.2f}%)")

    return total_water_pixels, total_background_pixels, total_water_tiles, total_background_tiles


totals = analyze_flood_images(flood_paths)
total_water_pixels, total_background_pixels, total_water_tiles, total_background_tiles = totals

7. Create a bar chart to visualize the distribution of two classes: 'Water Pixels' and 'Background Pixels'.
8. Create a bar chart to visualize the distribution of two classes: 'Water Tiles' and 'Background Tiles'.

What are the implication of an imbalanced dataset? How can we handle it? Let's plot the distribution excluding the `blank` images (all white VV). Let's recap which info we have:

In [ ]:
import matplotlib.pyplot as plt

def plot_class_distribution(counts, labels, title, ax=None):
    """Plot a labeled bar chart with percentage annotations."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 4))

    bars = ax.bar(labels, counts)
    total = sum(counts)

    for bar, count in zip(bars, counts):
        pct = 100 * count / total
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{pct:.1f}%",
            ha="center",
            va="bottom",
        )

    ax.set_title(title)
    ax.set_xlabel("Class")
    ax.set_ylabel("Count")
    return ax


# Side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))


plot_class_distribution(
    counts=[total_water_tiles, total_background_tiles],
    labels=["Water", "Background"],
    title="Class Distribution (tile level)",
    ax=axes[0],
)
plot_class_distribution(
    counts=[total_water_pixels, total_background_pixels],
    labels=["Water", "Background"],
    title="Class Distribution (pixel level)",
    ax=axes[1],
)

plt.tight_layout()
plt.show()

9. Generate a list of images without any empty images/labels and create a bar chart to visualize the distribution of the two classes: 'Water Tiles' and 'Background Tiles'.

In [ ]:
def clean_and_split_vv_images(vv_paths: List[str], empty_vv_paths: List[str], empty_flood_label_paths: List[str]) -> Tuple[List[str], List[str], List[str]]:
    """
    Clean and split VV images into background and water tiles.

    Args:
        vv_paths: List of VV image paths.
        empty_vv_paths: List of empty VV image paths.
        empty_flood_label_paths: List of empty flood label paths.

    Returns:
        vv_paths_cleaned: List of cleaned VV image paths.
        background_tiles: List of background tiles.
        water_tiles: List of water tiles.
    """
    # Normalize all inputs to strings for consistent comparison
    vv_paths = [str(p) for p in vv_paths]
    empty_vv_set = {str(p) for p in empty_vv_paths}
    empty_flood_set = {str(p) for p in empty_flood_label_paths}

    # List of VV images that are not blank
    vv_paths_cleaned = [d for d in vv_paths if d not in empty_vv_set]

    # Split into background and water tiles
    background_tiles = [
        d for d in vv_paths_cleaned
        if d.replace('/vv/', '/flood_label/').replace('_vv.png', '.png') in empty_flood_set
    ]
    water_tiles = [
        d for d in vv_paths_cleaned
        if d.replace('/vv/', '/flood_label/').replace('_vv.png', '.png') not in empty_flood_set
    ]

    # Ensure the split is correct
    assert len(background_tiles) + len(water_tiles) == len(vv_paths_cleaned)

    # Print the results
    print("Number of vv_paths_cleaned:", len(vv_paths_cleaned))
    print("Number of background_tiles:", len(background_tiles))
    print("Number of water_tiles:", len(water_tiles))

    # Save the results to CSV files
    np.savetxt('background_tiles.csv', background_tiles, fmt='%s', delimiter=",")
    np.savetxt('water_tiles.csv', water_tiles, fmt='%s', delimiter=",")
    print("Saved background_tiles.csv and water_tiles.csv")
    print("Cleaning and splitting completed.")
    return vv_paths_cleaned, background_tiles, water_tiles

In [ ]:
vv_paths_cleaned, background_tiles, water_tiles =clean_and_split_vv_images(vv_paths, empty_vv_paths, empty_flood_label_paths)

In [ ]:
# Define the class labels and counts
classes = ['Water Tiles', 'Background Tiles']

counts = [np.size(water_tiles), np.size(background_tiles)]  
# Create a bar chart
plt.bar(classes, counts)

total = sum(counts)
percentages = [(count/total)*100 for count in counts]


# Add a title and axis labels
plt.title('Class Distribution at the tile level')
plt.xlabel('Class')
plt.ylabel('Count')

for i, count in enumerate(counts):
    percentage = (count/total)*100
    plt.text(i, count+10, f"{percentage:.1f}%", ha='center')

# Show the plot
plt.show()

The plots show that the dataset is imbalanced. We will tackle this probelm in Lab_02. 

### Source
https://nasa-impact.github.io/etci2021/

https://github.com/sidgan/ETCI-2021-Competition-on-Flood-Detection